# Task 4 -- Join, Transformation Rules and Testing

## Objective

This notebook applies transformation rules to the validated dev.to dataset produced in
Task 3 (`data/interim/validated.csv`) and tests those rules before saving the
analysis-ready output.

Note on "Join": for this source, the "join" step already happened in Task 1 -- the extract
step pulls articles across several tags (`machinelearning`, `datascience`, `cloud`, `aws`,
`ai`) into one unified raw dataset, and duplicates are already dropped by URL in Task 2. The
join *across* group members' sources (dev.to + the other sources) happens later at the
project-wide integration stage, once everyone's `final.csv` is ready.

## Transformation Rules

| Rule | Transformation | Input Column(s) | Output Column | Description |
|------|----------------|-----------------|----------------|-------------|
| **R1** | Calculate Word Count | `content_clean` | `word_count` | Word count of the cleaned, plain-text article body. |
| **R2** | Handle Missing Topic | `topic` | `topic` | Replace missing/blank `topic` values with `"Uncategorized"`. |
| **R3** | Identify Long-Form Content | `word_count` | `is_long_form` | `True` when `word_count > 500`, else `False`. |
| **R4** | Engagement Score | `reactions_count`, `comments_count` | `engagement_score` | Sum of reactions and comments, as a simple engagement metric. |

R1 and R3 use the same column names and the same 500-word threshold as the Pluralsight
source, so the two datasets line up when they're combined at integration.

The final output is:

`../data/processed/final.csv`

## Imports and Data Ingestion

In [1]:
import os

import pandas as pd

INPUT_VALIDATED_PATH = "../data/interim/validated.csv"
OUTPUT_FINAL_PATH = "../data/processed/final.csv"

if not os.path.exists(INPUT_VALIDATED_PATH):
    raise FileNotFoundError(f"Missing required input file from Task 3: {INPUT_VALIDATED_PATH}")

df_validated = pd.read_csv(INPUT_VALIDATED_PATH)
print(f"Loaded validated data successfully. Shape: {df_validated.shape}")

Loaded validated data successfully. Shape: (275, 17)


## Transformation Rules Implementation

In [2]:
df_transformed = df_validated.copy()

# R1: word count, from the cleaned plain-text content
df_transformed["word_count"] = (
    df_transformed["content_clean"].fillna("").astype(str).apply(lambda x: len(x.split()))
)

# R2: fill missing/blank topic with "Uncategorized"
df_transformed["topic"] = (
    df_transformed["topic"]
    .astype(str)
    .str.strip()
    .replace({"": "Uncategorized", "nan": "Uncategorized", "None": "Uncategorized"})
)

# R3: long-form flag (same 500-word threshold used for the Pluralsight source)
df_transformed["is_long_form"] = df_transformed["word_count"] > 500

# R4: engagement score (dev.to-specific: reactions + comments)
df_transformed["engagement_score"] = (
    df_transformed["reactions_count"].fillna(0) + df_transformed["comments_count"].fillna(0)
)

print("Transformation rules successfully applied to data columns.")

Transformation rules successfully applied to data columns.


## Unit Testing & Assertions

In [3]:
print("=" * 73)
print("RUNNING PIPELINE QUALITY CONTROL TEST ASSERTIONS")
print("=" * 73)

# Test 1: word counts are non-negative
assert (df_transformed["word_count"] >= 0).all(), "Test Fail: found negative word counts!"
print("Test 1 Passed: all word counts are valid non-negative integers.")

# Test 2: no blank topic values remain
assert not (df_transformed["topic"].astype(str).str.strip() == "").any(), (
    "Test Fail: found blank topic values!"
)
print("Test 2 Passed: topic imputation verified successfully.")

# Test 3: long-form flag lines up with the word-count threshold
short_form = df_transformed[df_transformed["word_count"] <= 500]
assert not short_form["is_long_form"].any(), (
    "Test Fail: flagged long-form content incorrectly below threshold!"
)
print("Test 3 Passed: long-form flag validated against the word-count threshold.")

# Test 4: engagement score is never negative
assert (df_transformed["engagement_score"] >= 0).all(), (
    "Test Fail: found a negative engagement score!"
)
print("Test 4 Passed: engagement score is valid.")

print("\nALL PIPELINE ASSERTION TESTS PASSED SUCCESSFULLY!")

RUNNING PIPELINE QUALITY CONTROL TEST ASSERTIONS
Test 1 Passed: all word counts are valid non-negative integers.
Test 2 Passed: topic imputation verified successfully.
Test 3 Passed: long-form flag validated against the word-count threshold.
Test 4 Passed: engagement score is valid.

ALL PIPELINE ASSERTION TESTS PASSED SUCCESSFULLY!


## Save `final.csv`

In [4]:
os.makedirs(os.path.dirname(OUTPUT_FINAL_PATH), exist_ok=True)

df_transformed.to_csv(OUTPUT_FINAL_PATH, index=False, encoding="utf-8")

print(f"\nPipeline complete! Final dataset saved to: {OUTPUT_FINAL_PATH}")
print(f"Final exported row count: {df_transformed.shape}")
print("\nColumns included in the final delivery file:")
print(list(df_transformed.columns))


Pipeline complete! Final dataset saved to: ../data/processed/final.csv
Final exported row count: (275, 20)

Columns included in the final delivery file:
['external_id', 'title', 'source', 'author', 'published_date', 'url', 'topic', 'tags', 'description', 'content_type', 'reading_time_minutes', 'reactions_count', 'comments_count', 'cover_image', 'content_markdown', 'content_html', 'content_clean', 'word_count', 'is_long_form', 'engagement_score']
